In [1]:
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.server import SimpleXMLRPCRequestHandler
import sqlite3
import datetime
#import os

#import glob
#import re

In [2]:
PORT = 8038
DB_FILE = 'log.db'

In [3]:
class RequestHandler(SimpleXMLRPCRequestHandler):
    rpc_paths = ('/RPC2',)

server = SimpleXMLRPCServer(("localhost", PORT),
                            requestHandler=RequestHandler,
                            logRequests=True)

In [4]:
# Подключение к БД
conn = sqlite3.connect(DB_FILE, check_same_thread=False)

# Создание таблицы, если её нет
conn.execute('''
    CREATE TABLE IF NOT EXISTS logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        event_type TEXT NOT NULL,
        timestamp TEXT NOT NULL,
        duration_ms REAL
    )
''')
conn.commit()

#### Функции работы с таблицей логов

In [7]:
def add_log(event_type, duration_ms=0.0):
    """ Добавляет событие в журнал. """
    
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    conn.execute(
        'INSERT INTO logs (event_type, timestamp, duration_ms) VALUES (?, ?, ?)',
        (event_type, timestamp, duration_ms)
    )
    conn.commit()
    
    return True

server.register_function(add_log, 'add_log')

<function __main__.add_log(event_type, duration_ms=0.0)>

In [8]:
def get_log():
    """
    Возвращает весь журнал событий.
    Формат: список кортежей [(event_type, timestamp, duration_ms), ...]
    """
    cursor = conn.execute(
        'SELECT event_type, timestamp, duration_ms FROM logs ORDER BY id'
    )
    return cursor.fetchall()

server.register_function(get_log, 'get_log')

<function __main__.get_log()>

In [9]:
def get_log_by_type(event_type):
    """
    Возвращает срез журнала по типу события.
    Учитывает базовый тип (до двоеточия), если в event_type есть детали.
    """
    cursor = conn.execute(
        '''SELECT event_type, timestamp, duration_ms 
           FROM logs 
           WHERE event_type = ? OR event_type LIKE ?
           ORDER BY id''',
        (event_type, f"{event_type}:%")
    )
    return cursor.fetchall()

server.register_function(get_log_by_type, 'get_log_by_type')

<function __main__.get_log_by_type(event_type)>

In [10]:
def get_log_by_time(start_time, end_time):
    """
    Возвращает срез журнала по времени.
    start_time и end_time — строки в формате 'YYYY-MM-DD HH:MM:SS'
    """
    cursor = conn.execute(
        '''SELECT event_type, timestamp, duration_ms 
           FROM logs 
           WHERE timestamp >= ? AND timestamp <= ?
           ORDER BY id''',
        (start_time, end_time)
    )
    return cursor.fetchall()

server.register_function(get_log_by_time, 'get_log_by_time')

<function __main__.get_log_by_time(start_time, end_time)>

In [11]:
def get_log_by_duration(min_ms, max_ms):
    """
    Возвращает срез журнала по длительности выполнения.
    min_ms и max_ms — границы длительности в миллисекундах.
    """
    cursor = conn.execute(
        '''SELECT event_type, timestamp, duration_ms 
           FROM logs 
           WHERE duration_ms >= ? AND duration_ms <= ?
           ORDER BY id''',
        (min_ms, max_ms)
    )
    return cursor.fetchall()

server.register_function(get_log_by_duration, 'get_log_by_duration')

<function __main__.get_log_by_duration(min_ms, max_ms)>

In [12]:
def get_log_count():
    """Возвращает общее количество записей в журнале."""
    cursor = conn.execute('SELECT COUNT(*) FROM logs')
    return cursor.fetchone()[0]

server.register_function(get_log_count, 'get_log_count')

<function __main__.get_log_count()>

In [13]:
def clear_log():
    """Очищает журнал событий (для тестирования)."""
    conn.execute('DELETE FROM logs')
    conn.commit()
    return True

server.register_function(clear_log, 'clear_log')

<function __main__.clear_log()>

#### Запуск сервера

In [ ]:
print(f"Listening on port {PORT}...")
print(f"Сервер статистики 2 готов к работе")

try:
    server.serve_forever()
except KeyboardInterrupt:
    print("\nОстановка сервера...")
    conn.close()
    print("Соединение с БД закрыто")

Listening on port 8038...
Сервер статистики 2 готов к работе


127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:50] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:55] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 21:42:59] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 22:22:22] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 22:22:22] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/2026 22:22:23] "POST /RPC2 HTTP/1.1" 200 -
127.0.0.1 - - [18/Oct/202